# DGS-Fingeralphabet: Training und Evaluation

Dieses Notebook ist der Python-Teil der Pipeline:

1. **Browser**: Aufnahme-Modus speichert Handpunkte (Landmarks) als JSON in `data/raw/`
2. **Notebook**: Features, Training, ehrliche Evaluation ← *hier*
3. **Browser**: Das trainierte Modell wird als JSON exportiert und in der App ausgeführt

Leitfrage für die Evaluation: *Wie gut erkennt das Modell die Hand einer Person, die es beim Training nie gesehen hat?*

## Schritt 1: Daten laden und verstehen

Jede Datei in `data/raw/` ist ein Export aus dem Aufnahme-Modus. Ein **Sample** ist ein einzelnes Kamerabild:

```json
{ "letter": "A", "person": "user1", "features": [63 Zahlen] }
```

Die 63 Zahlen sind **21 Handpunkte × (x, y, z)**, so wie MediaPipe sie liefert. Der Browser hat sie bereits normalisiert
(siehe `src/features.ts`, Funktion `normalize`):

- **Verschiebung:** Das Handgelenk (Punkt 0) liegt im Ursprung.
- **Skalierung:** Der Abstand Handgelenk → Mittelfinger-Grundgelenk (Punkt 9) ist 1. Große und kleine Hände werden vergleichbar.
- **Spiegelung:** Linke Hände werden auf rechte gespiegelt.

Die Ausrichtung zur Kamera steckt dagegen noch in den Zahlen. Das wird in Schritt 2 wichtig.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA = Path('../data/raw')

rows = []
for file in sorted(DATA.glob('*.json')):
    file_rows = json.loads(file.read_text())
    print(f'{file.name}: {len(file_rows)} Samples')
    rows += file_rows
print(f'zusammen: {len(rows)} Samples')

### Duplikate entfernen

Ein Export enthält **alle** Aufnahmen des Browsers, nicht nur die der Person im Dateinamen. Wurden zwei Personen im
selben Browser aufgenommen, stehen ihre Samples in mehreren Dateien. Ohne Filter würden sie doppelt zählen, und das
Modell würde diese Samples beim Training stärker gewichten.

Als Schlüssel dient das Tripel (Person, Buchstabe, Features). Zwei echte Kamerabilder ergeben praktisch nie exakt
dieselben 63 Zahlen. Ein paar Duplikate *innerhalb* einer Datei sind trotzdem möglich: Zeigt ein Video dasselbe Bild
zweimal hintereinander, liefert MediaPipe zweimal dieselben Punkte. Auch die fliegen raus, denn sie bringen keine
neue Information.

In [ ]:
seen = set()
samples = []
for s in rows:
    key = (s['person'], s['letter'], tuple(s['features']))
    if key not in seen:
        seen.add(key)
        samples.append(s)
print(f'{len(rows) - len(samples)} Duplikate entfernt, {len(samples)} Samples bleiben')

### Als Arrays

Für scikit-learn braucht es die übliche Form:

- `X`: eine Zeile pro Sample, eine Spalte pro Feature → Form `(Anzahl Samples, 63)`
- `y`: das Label (der Buchstabe) pro Sample
- `persons`: wer die Hand gezeigt hat. Das ist kein Feature, das Modell bekommt es nie zu sehen. Wir brauchen es nur,
  um bei der Evaluation nach Personen zu trennen.

In [ ]:
X_raw = np.array([s['features'] for s in samples])
y = np.array([s['letter'] for s in samples])
persons = np.array([s['person'] for s in samples])

print('X_raw:', X_raw.shape, '  y:', y.shape, '  persons:', persons.shape)
print('Buchstaben:', ''.join(sorted(set(y))))
print('Personen:', sorted(set(persons.tolist())))

### Überblick: Wie viele Samples pro Buchstabe und Person?

Darauf achten:

- Fehlen Buchstaben bei einer Person?
- Sind die Mengen ähnlich groß? Hat eine Person viel mehr Daten, dominiert sie beim Training.
- Pro Durchgang sind es bis zu 60 Frames. Frames aus *einem* Durchgang sind sich sehr ähnlich, viele Frames heißen
  also nicht automatisch viel Abwechslung.

In [ ]:
counts = pd.crosstab(pd.Series(y, name='Buchstabe'), pd.Series(persons, name='Person'), margins=True, margins_name='Summe')
counts

### Eine Hand ansehen

Zahlen allein sagen wenig, also zeichnen wir die **mittlere Hand** pro Buchstabe und Person: den Mittelwert aller
Samples, als Skelett von vorne gesehen (x/y; die Tiefe z wird ignoriert).

Die Punkte sind so verbunden wie in der App: vom Handgelenk über die vier Punkte jedes Fingers zur Fingerspitze.
Die y-Achse zeigt wie bei Bildkoordinaten nach unten, deshalb wird sie zum Zeichnen umgedreht.

In [ ]:
FINGERS = [[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16], [17, 18, 19, 20]]
COLORS = ['tab:orange', 'tab:blue', 'tab:green', 'tab:red', 'tab:purple']  # Daumen ... kleiner Finger

def draw_hand(ax, features, title):
    pts = np.asarray(features).reshape(21, 3)  # 63 Zahlen -> 21 Punkte mit (x, y, z)
    for chain, color in zip(FINGERS, COLORS):
        idx = [0] + chain  # vom Handgelenk aus
        ax.plot(pts[idx, 0], -pts[idx, 1], '-o', color=color, markersize=3)
    ax.set_title(title)
    ax.set_aspect('equal')
    ax.set_xlim(-1.4, 1.0)
    ax.set_ylim(-0.3, 1.5)
    ax.axis('off')

def mean_hand(letter, person):
    return X_raw[(y == letter) & (persons == person)].mean(axis=0)

letters = ['A', 'S', 'M', 'N', 'G', 'H']
people = sorted(set(persons.tolist()))
fig, axes = plt.subplots(len(people), len(letters), figsize=(2.2 * len(letters), 2.6 * len(people)))
for r, person in enumerate(people):
    for c, letter in enumerate(letters):
        draw_hand(axes[r, c], mean_hand(letter, person), f'{letter} – {person}')
plt.tight_layout()

**TODO:**

1. Sehen A und S bei *einer* Person ähnlich aus? Und M und N? Das sind die Paare, die in der App verwechselt werden.
2. Vergleiche denselben Buchstaben zwischen den Personen, etwa M oben und M unten. Liegt der Unterschied in der
   *Fingerstellung* oder darin, wie die *ganze Hand gedreht* ist?
3. G und H zeigen zur Seite. Bei diesen Buchstaben ist die Ausrichtung selbst ein Merkmal.

Frage 2 führt direkt zu Schritt 2.